# Smart Recycle Assistant – Huấn luyện mô hình phân loại rác (MobileNetV2)

Chạy trên **Google Colab** (Runtime → Change runtime type → **T4 GPU**). Chạy lần lượt từng ô từ trên xuống.

Quy trình: tải dữ liệu → ánh xạ nhãn về 8 nhóm → làm sạch (xóa ảnh lỗi/nhỏ/trùng bằng pHash) → chia Train/Val/Test 70/15/15 → huấn luyện 2 giai đoạn (Transfer Learning rồi Fine-tuning) → đánh giá trên tập Test → lưu mô hình.

**Việc bạn phải sửa:** ô 2 (danh sách 8 nhóm `CLASSES` và bảng ánh xạ `LABEL_MAP`) cho đúng với đề tài của nhóm.

## 1. Cài đặt

In [ ]:
!pip -q install imagehash scikit-learn
import os, json, shutil, random, hashlib, time
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Cấu hình 8 nhóm rác và bảng ánh xạ nhãn  ⚠️ SỬA Ô NÀY

- `CLASSES`: 8 nhóm mục tiêu của đồ án. **Danh sách dưới đây chỉ là ví dụ**, hãy thay bằng 8 nhóm thật của nhóm.
- `LABEL_MAP`: tên thư mục gốc trong từng bộ dữ liệu (viết thường) → nhóm của đồ án. Thư mục nào không có trong bảng (và không trùng tên nhóm) sẽ bị bỏ qua và in ra để bạn biết.
- Ảnh tự chụp ở Việt Nam: đặt vào `data/raw/vietnam/<tên_nhóm>/...` với tên thư mục đúng bằng tên nhóm trong `CLASSES`.

In [ ]:
CLASSES = ["giay", "nhua", "kim_loai", "thuy_tinh", "huu_co", "pin_nguy_hai", "rac_khac"]

LABEL_MAP = {
    # TrashNet + Kaggle 12 lớp
    "paper": "giay", "cardboard": "giay",
    "plastic": "nhua",
    "metal": "kim_loai",
    "glass": "thuy_tinh", "brown-glass": "thuy_tinh", "green-glass": "thuy_tinh", "white-glass": "thuy_tinh",
    "biological": "huu_co",
    "battery": "pin_nguy_hai", "batteries": "pin_nguy_hai",
    "trash": "rac_khac",
}

IMG_SIZE = (224, 224)
BATCH = 32
RAW_DIR = Path("data/raw")
ALL_DIR = Path("data/all")
SPLIT_DIR = Path("data/split")
MIN_BYTES = 2 * 1024          # ảnh nhỏ hơn 2KB bị loại (số thật là 2 * 1024)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## 3. Tải dữ liệu

Cần file `kaggle.json` (kaggle.com → Settings → API → Create New Token). Chạy ô dưới, chọn file để upload.
Có thể bỏ qua bộ nào không muốn dùng. Ảnh tự chụp: upload vào `data/raw/vietnam/<tên_nhóm>/`.

In [ ]:
from google.colab import files
files.upload()   # chọn kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d feyzazkefe/trashnet -p data/raw/trashnet --unzip -q
!kaggle datasets download -d mostafaabla/garbage-classification -p data/raw/kaggle12 --unzip -q
# Muốn dùng bản đã sửa nhãn/bỏ trùng thì thay bằng: huberthamelin/garbage-classification-labels-corrections
!mkdir -p data/raw/vietnam
!find data/raw -maxdepth 3 -type d | head -50

Dataset URL: https://www.kaggle.com/datasets/feyzazkefe/trashnet
License(s): unknown
Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
data/raw
data/raw/vietnam
data/raw/kaggle12
data/raw/kaggle12/garbage_classification
data/raw/kaggle12/garbage_classification/clothes
data/raw/kaggle12/garbage_classification/brown-glass
data/raw/kaggle12/garbage_classification/battery
data/raw/kaggle12/garbage_classification/biological
data/raw/kaggle12/garbage_classification/plastic
data/raw/kaggle12/garbage_classification/white-glass
data/raw/kaggle12/garbage_classification/paper
data/raw/kaggle12/garbage_classification/green-glass
data/raw/kaggle12/garbage_classification/cardboard
data/raw/kaggle12/garbage_classification/trash
data/raw/kaggle12/garbage_classification/metal
data/raw/kaggle12/garbage_classification/shoes
data/raw/trashnet
data/raw/trashnet/dataset-resized
data/raw/trashnet/dataset-resized/glass
data/raw/trashnet/dataset-resized/plasti

## 4. Gộp về 8 nhóm + làm sạch (xóa ảnh lỗi, quá nhỏ, trùng pHash)

Ảnh trùng được phát hiện bằng Perceptual Hash (pHash): hai ảnh có hash giống hệt nhau thì giữ một.
Làm bước này **trước khi chia tập** để cùng một ảnh không lọt vào cả Train lẫn Test.

In [ ]:
if ALL_DIR.exists():
    shutil.rmtree(ALL_DIR)
for c in CLASSES:
    (ALL_DIR / c).mkdir(parents=True, exist_ok=True)

skipped_folders = {}
stats = dict(copied=0, too_small=0, corrupt=0, duplicate=0)
seen_hashes = set()

all_files = [p for p in RAW_DIR.rglob("*") if p.suffix.lower() in IMG_EXT]
random.shuffle(all_files)

for p in all_files:
    key = p.parent.name.lower()
    target = LABEL_MAP.get(key, key if key in CLASSES else None)
    if target is None:
        skipped_folders[key] = skipped_folders.get(key, 0) + 1
        continue
    if p.stat().st_size < MIN_BYTES:
        stats["too_small"] += 1
        continue
    try:
        with Image.open(p) as im:
            im = im.convert("RGB")
            h = str(imagehash.phash(im))
    except Exception:
        stats["corrupt"] += 1
        continue
    if h in seen_hashes:
        stats["duplicate"] += 1
        continue
    seen_hashes.add(h)
    out = ALL_DIR / target / f"{hashlib.md5(str(p).encode()).hexdigest()}.jpg"
    im.save(out, quality=95)
    stats["copied"] += 1

print("Kết quả làm sạch:", stats)
print("Thư mục bị bỏ qua vì không có trong LABEL_MAP:", skipped_folders)
print({c: len(list((ALL_DIR / c).glob('*.jpg'))) for c in CLASSES})

Kết quả làm sạch: {'copied': 8174, 'too_small': 160, 'corrupt': 0, 'duplicate': 2406}
Thư mục bị bỏ qua vì không có trong LABEL_MAP: {'clothes': 5325, 'shoes': 1977}
{'giay': 1930, 'nhua': 858, 'kim_loai': 766, 'thuy_tinh': 1971, 'huu_co': 981, 'pin_nguy_hai': 936, 'tui_ni_long_xop': 0, 'rac_khac': 732}


## 5. Chia Train / Val / Test = 70 / 15 / 15 (chia đều theo từng nhóm)

Tập Test được giữ riêng, chỉ dùng một lần ở cuối để đo độ chính xác.

In [ ]:
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

for c in CLASSES:
    files_c = sorted((ALL_DIR / c).glob("*.jpg"))
    if len(files_c) < 10:
        print(f"CẢNH BÁO: nhóm '{c}' chỉ có {len(files_c)} ảnh, cần thu thập thêm.")
        if len(files_c) < 3:
            continue
    train_f, rest_f = train_test_split(files_c, test_size=0.30, random_state=SEED)
    val_f, test_f = train_test_split(rest_f, test_size=0.50, random_state=SEED)
    for name, fl in [("train", train_f), ("val", val_f), ("test", test_f)]:
        d = SPLIT_DIR / name / c
        d.mkdir(parents=True, exist_ok=True)
        for f in fl:
            shutil.copy(f, d / f.name)

for s in ["train", "val", "test"]:
    print(s, {c: len(list((SPLIT_DIR / s / c).glob('*.jpg'))) for c in CLASSES if (SPLIT_DIR / s / c).exists()})

train {'giay': 1351, 'nhua': 600, 'kim_loai': 536, 'thuy_tinh': 1379, 'huu_co': 686, 'pin_nguy_hai': 655, 'rac_khac': 512}
val {'giay': 289, 'nhua': 129, 'kim_loai': 115, 'thuy_tinh': 296, 'huu_co': 147, 'pin_nguy_hai': 140, 'rac_khac': 110}
test {'giay': 290, 'nhua': 129, 'kim_loai': 115, 'thuy_tinh': 296, 'huu_co': 148, 'pin_nguy_hai': 141, 'rac_khac': 110}


## 6. Nạp dữ liệu

- Resize về 224×224 (kích thước đầu vào của MobileNetV2).
- Chuẩn hóa pixel **nằm trong mô hình** (`Rescaling(1/127.5, offset=-1)` → [-1, 1], đúng chuẩn MobileNetV2), nên khi dự đoán chỉ cần đưa ảnh thô 0–255 vào.
- Augmentation cũng nằm trong mô hình và **chỉ chạy khi huấn luyện**, không áp dụng cho Val/Test.

In [ ]:
def load(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split, labels="inferred", label_mode="categorical",
        class_names=CLASSES, image_size=IMG_SIZE, batch_size=BATCH,
        shuffle=shuffle, seed=SEED)

train_ds = load("train", True)
val_ds = load("val", False)
test_ds = load("test", False)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Trọng số lớp để bù khi các nhóm lệch số lượng ảnh
counts = np.array([len(list((SPLIT_DIR / "train" / c).glob("*.jpg"))) for c in CLASSES], dtype=float)
class_weight = {i: float(counts.sum() / (len(CLASSES) * max(n, 1))) for i, n in enumerate(counts)}
print("class_weight:", {CLASSES[i]: round(w, 2) for i, w in class_weight.items()})

Found 5719 files belonging to 7 classes.
Found 1226 files belonging to 7 classes.
Found 1229 files belonging to 7 classes.
class_weight: {'giay': 0.6, 'nhua': 1.36, 'kim_loai': 1.52, 'thuy_tinh': 0.59, 'huu_co': 1.19, 'pin_nguy_hai': 1.25, 'rac_khac': 1.6}


## 7. Xây dựng mô hình (Transfer Learning)

Đóng băng MobileNetV2 (đã học trên ImageNet), chỉ huấn luyện phần phân loại ở đầu ra.
Loss: Categorical Crossentropy. Optimizer: Adam. LR được giảm tự động khi val loss không cải thiện.

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.125),          # 0.125 vòng = ±45°
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.2, 0.2),    # dịch ngang/dọc 20%
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.2),
], name="augmentation")

base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,), name="image_0_255")
x = augment(inputs)
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(x)   # [0,255] -> [-1,1]
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_0_255 (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 8. Huấn luyện giai đoạn 1 (chỉ phần đầu ra)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True),
]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=callbacks)

Epoch 1/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 30s 97ms/step - accuracy: 0.6672 - loss: 0.9372 - val_accuracy: 0.8475 - val_loss: 0.4836 - learning_rate: 0.0010
Epoch 2/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 14s 80ms/step - accuracy: 0.7993 - loss: 0.5661 - val_accuracy: 0.8777 - val_loss: 0.4075 - learning_rate: 0.0010
Epoch 3/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - accuracy: 0.8293 - loss: 0.4877 - val_accuracy: 0.8842 - val_loss: 0.3810 - learning_rate: 0.0010
Epoch 4/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 14s 79ms/step - accuracy: 0.8362 - loss: 0.4568 - val_accuracy: 0.8825 - val_loss: 0.3702 - learning_rate: 0.0010
Epoch 5/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 21s 81ms/step - accuracy: 0.8405 - loss: 0.4502 - val_accuracy: 0.8899 - val_loss: 0.3641 - learning_rate: 0.0010
Epoch 6/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 15s 82ms/step - accuracy: 0.8407 - loss: 0.4294 - val_accuracy: 0.8907 - val_loss: 0.3402 - learning_rate: 0.0010
Epoch 7/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 14s 80ms/step - accuracy: 0.8482 - l

## 9. Giai đoạn 2: Fine-tuning

Mở khóa khoảng 30 lớp cuối của MobileNetV2 và học với learning rate rất nhỏ để mô hình thích nghi với ảnh rác.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=callbacks)

Epoch 1/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 31s 106ms/step - accuracy: 0.7968 - loss: 0.5791 - val_accuracy: 0.8931 - val_loss: 0.3311 - learning_rate: 1.0000e-05
Epoch 2/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 18s 103ms/step - accuracy: 0.8192 - loss: 0.4966 - val_accuracy: 0.9005 - val_loss: 0.3207 - learning_rate: 1.0000e-05
Epoch 3/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 19s 103ms/step - accuracy: 0.8292 - loss: 0.4646 - val_accuracy: 0.9029 - val_loss: 0.3197 - learning_rate: 1.0000e-05
Epoch 4/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 20s 99ms/step - accuracy: 0.8456 - loss: 0.4310 - val_accuracy: 0.9005 - val_loss: 0.3185 - learning_rate: 1.0000e-05
Epoch 5/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 19s 106ms/step - accuracy: 0.8486 - loss: 0.4088 - val_accuracy: 0.9062 - val_loss: 0.3159 - learning_rate: 1.0000e-05
Epoch 6/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 18s 101ms/step - accuracy: 0.8692 - loss: 0.3749 - val_accuracy: 0.9070 - val_loss: 0.3115 - learning_rate: 1.0000e-05
Epoch 7/15
179/179 ━━━━━━━━━━━━━━━━━━━━ 19s 104

## 10. Đánh giá trên tập Test (chỉ chạy sau khi đã huấn luyện xong)

Mục tiêu của đồ án: độ chính xác ≥ 85%. Xem thêm Precision/Recall từng nhóm để biết nhóm nào còn yếu.

In [ ]:
model = tf.keras.models.load_model("best_model.keras")
y_true, y_pred = [], []
for xb, yb in test_ds:
    y_true.extend(np.argmax(yb.numpy(), axis=1))
    y_pred.extend(np.argmax(model.predict(xb, verbose=0), axis=1))

acc = float(np.mean(np.array(y_true) == np.array(y_pred)))
print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_true, y_pred, labels=list(range(len(CLASSES))), target_names=CLASSES, zero_division=0))
print("Confusion matrix (hàng = thật, cột = dự đoán):")
print(confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES)))))

Test accuracy: 0.9056
              precision    recall  f1-score   support

        giay       0.97      0.92      0.95       290
        nhua       0.76      0.85      0.80       129
    kim_loai       0.76      0.90      0.83       115
   thuy_tinh       0.97      0.86      0.91       296
      huu_co       0.95      0.97      0.96       148
pin_nguy_hai       0.95      0.96      0.96       141
    rac_khac       0.84      0.86      0.85       110

    accuracy                           0.91      1229
   macro avg       0.89      0.91      0.89      1229
weighted avg       0.91      0.91      0.91      1229

Confusion matrix (hàng = thật, cột = dự đoán):
[[268   4   7   0   2   2   7]
 [  2 110   4   6   0   0   7]
 [  0   6 104   2   0   3   0]
 [  1  15  17 256   3   2   2]
 [  0   2   0   1 144   0   1]
 [  0   1   3   0   0 136   1]
 [  4   7   2   0   2   0  95]]


## 11. Đo thời gian dự đoán một ảnh (yêu cầu < 3 giây)

In [ ]:
dummy = np.random.randint(0, 255, (1,) + IMG_SIZE + (3,)).astype("float32")
model.predict(dummy, verbose=0)  # khởi động
t0 = time.time()
for _ in range(20):
    model.predict(dummy, verbose=0)
print(f"Thời gian trung bình mỗi ảnh: {(time.time() - t0) / 20 * 1000:.1f} ms")

Thời gian trung bình mỗi ảnh: 83.0 ms


## 12. Lưu mô hình và hàm dự đoán cho backend

`labels.json` lưu thứ tự nhóm; backend dùng đúng thứ tự này khi đọc kết quả. Mô hình nhận ảnh thô 224×224, giá trị 0–255.

In [ ]:
model.save("smart_recycle_model.keras")
with open("labels.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, ensure_ascii=False)

def predict_image(path, top_k=3):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img, dtype="float32"), 0)
    probs = model.predict(arr, verbose=0)[0]
    idx = np.argsort(probs)[::-1][:top_k]
    return [{"label": CLASSES[i], "confidence": float(probs[i])} for i in idx]

# Ví dụ: predict_image("data/split/test/nhua/<ten_anh>.jpg")

# Xuất TFLite (tùy chọn, cho ứng dụng di động)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
open("smart_recycle_model.tflite", "wb").write(converter.convert())

from google.colab import files
files.download("smart_recycle_model.keras")
files.download("labels.json")

Saved artifact at '/tmp/tmpm_dku434'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image_0_255')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  133268760315088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760317776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760317584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760317968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760314512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760318160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760318928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760318736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760317392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760314704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133268760318352

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip -q install onnx onnxruntime tf2onnx
import numpy as np, tensorflow as tf, onnxruntime as ort
from google.colab import files

model = tf.keras.models.load_model("smart_recycle_model.keras")

# Dựng model chỉ để dự đoán: bỏ lớp input cũ và lớp augmentation (ONNX không hỗ trợ các phép ngẫu nhiên)
inp = tf.keras.Input(shape=(224, 224, 3))
x = inp
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.InputLayer) or "augment" in layer.name:
        continue
    x = layer(x)
infer = tf.keras.Model(inp, x)

# Kiểm tra: model mới phải cho kết quả giống model gốc
t = np.random.randint(0, 256, (1, 224, 224, 3)).astype("float32")
print("Keras mới vs gốc:", np.abs(infer(t, training=False).numpy() - model(t, training=False).numpy()).max())

infer.export("smart_recycle_model.onnx", format="onnx")
sess = ort.InferenceSession("smart_recycle_model.onnx")
name = sess.get_inputs()[0].name
out_onnx = sess.run(None, {name: t})[0]
print("ONNX vs Keras:", np.abs(out_onnx - infer(t, training=False).numpy()).max())
files.download("smart_recycle_model.onnx")

In [2]:
from pathlib import Path
import shutil
from google.colab import files

SPLIT_DIR = Path("data/split")
CLASSES = ["giay", "nhua", "kim_loai", "thuy_tinh", "huu_co", "pin_nguy_hai", "rac_khac"]

print("Có thư mục data/split:", SPLIT_DIR.exists())
if SPLIT_DIR.exists():
    for s in ["train", "val", "test"]:
        print(s, {c: len(list((SPLIT_DIR / s / c).glob("*.jpg"))) for c in CLASSES if (SPLIT_DIR / s / c).exists()})
    shutil.make_archive("test_split", "zip", "data/split", "test")
    files.download("test_split.zip")

Có thư mục data/split: False
